In [9]:
afc_countries = {
    1: "Afghanistan", 12: "Australia", 16: "Bahrain", 17: "Bangladesh",
    24: "Bhutan", 30: "Brunei", 34: "Cambodia", 42: "China",
    76: "Guam", 83: "Hong_Kong", 86: "India", 87: "Indonesia",
    88: "Iran", 89: "Iraq", 94: "Japan", 95: "Jordan",
    99: "Kuwait", 100: "Kyrgyzstan", 101: "Laos", 103: "Lebanon",
    110: "Macau", 114: "Malaysia", 115: "Maldives", 123: "Mongolia",
    128: "Nepal", 136: "North_Korea", 139: "Oman", 140: "Pakistan",
    141: "Palestine", 146: "Philippines", 150: "Qatar", 161: "Saudi_Arabia",
    167: "Singapore", 173: "South_Korea", 175: "Sri_Lanka", 181: "Syria",
    183: "Taiwan", 184: "Tajikistan", 186: "Thailand", 193: "Turkmenistan",
    197: "United_Arab_Emirates", 201: "Uzbekistan", 204: "Vietnam", 206: "Yemen",
    214: "Northern_Mariana_Islands", 229: "East_Timor", 230: "Myanmar"
}

In [ ]:
# Run scraper to test on Vietnam 2025 page

import re
from typing import Any, Dict, List, Optional
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup


def scrape_country_year_info(country_id: int, year: int, country_slug: str) -> Dict[str, Any]:
    """Scrape average height, average age, and match report IDs from national-football-teams."""
    base_url = "https://national-football-teams.com"
    page_url = f"{base_url}/country/{country_id}/{year}/{country_slug}.html"

    response = requests.get(page_url, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    def find_value_after_label(label_pattern: str) -> Optional[str]:
        # Find the <strong> label, then read value from its parent row.
        strong = soup.find("strong", string=re.compile(label_pattern, re.IGNORECASE))
        if not strong:
            return None
        row = strong.find_parent("div", class_="row")
        if not row:
            return None
        cols = row.find_all("div", class_=re.compile(r"\bcol-\d+\b"))
        if len(cols) >= 2:
            return cols[1].get_text(" ", strip=True) or None
        return None

    avg_height = find_value_after_label(r"Average\s+height\s+in\s+\d{4}")
    avg_age = find_value_after_label(r"Average\s+age\s+in\s+\d{4}")

    match_ids: List[int] = []
    seen = set()
    for a_tag in soup.select("a[href*='/matches/report/']"):
        href = a_tag.get("href", "")
        m = re.search(r"/matches/report/(\d+)/", href)
        if m:
            match_id = int(m.group(1))
            if match_id not in seen:
                seen.add(match_id)
                match_ids.append(match_id)

    return {
        "country_id": country_id,
        "country_slug": country_slug,
        "year": year,
        "page_url": page_url,
        "average_height": avg_height,
        "average_age": avg_age,
        "match_ids": match_ids,
        "match_count": len(match_ids),
    }


result = scrape_country_year_info(204, 2025, "Vietnam")
result

{'country_id': 204,
 'country_slug': 'Vietnam',
 'year': 2025,
 'page_url': 'https://national-football-teams.com/country/204/2025/Vietnam.html',
 'average_height': '1.77m',
 'average_age': '26.1',
 'match_ids': [40420, 40421, 41185, 41131, 41697, 42366, 42481, 42782],
 'match_count': 8}

In [ ]:
#Run a full scraper from https://national-football-teams.com

import random
import re
import time
from dataclasses import dataclass
from datetime import datetime
from typing import Dict, Tuple

import pandas as pd
import requests
from bs4 import BeautifulSoup


BASE_URL = "https://national-football-teams.com"


@dataclass
class ScrapeStop(Exception):
    """Signal to stop the whole scraping job due to critical HTTP/network errors."""
    message: str


def extract_country_year_info_from_html(html: str, page_url: str, country_id: int, country_slug: str, year: int):
    """Parse average height, average age, and unique match IDs from one country-year HTML page."""
    soup = BeautifulSoup(html, "html.parser")

    def find_value_after_label(label_pattern: str):
        strong = soup.find("strong", string=re.compile(label_pattern, re.IGNORECASE))
        if not strong:
            return None
        row = strong.find_parent("div", class_="row")
        if not row:
            return None
        cols = row.find_all("div", class_=re.compile(r"\bcol-\d+\b"))
        if len(cols) >= 2:
            value = cols[1].get_text(" ", strip=True)
            return value or None
        return None

    avg_height = find_value_after_label(r"Average\s+height\s+in\s+\d{4}")
    avg_age = find_value_after_label(r"Average\s+age\s+in\s+\d{4}")

    # Keep page-level IDs unique while preserving order.
    page_match_ids = []
    seen_page_ids = set()
    for a_tag in soup.select("a[href*='/matches/report/']"):
        href = a_tag.get("href", "")
        m = re.search(r"/matches/report/(\d+)/", href)
        if m:
            match_id = int(m.group(1))
            if match_id not in seen_page_ids:
                seen_page_ids.add(match_id)
                page_match_ids.append(match_id)

    # Treat pages with no key data and no match links as effectively empty for this task.
    has_data = bool(avg_height or avg_age or page_match_ids)

    row = {
        "country_id": country_id,
        "country_slug": country_slug,
        "year": year,
        "page_url": page_url,
        "average_height": avg_height,
        "average_age": avg_age,
        "match_count": len(page_match_ids),
        "match_ids": page_match_ids,
        "scraped_at_utc": datetime.utcnow().isoformat(timespec="seconds"),
    }
    return row, has_data, page_match_ids


def scrape_all_afc_country_years(
    countries: Dict[int, str],
    start_year: int = 1995,
    end_year: int = 2026,
    sleep_range: Tuple[float, float] = (2, 5),
    long_break_every: int = 50,
    long_break_seconds: int = 60,
    stop_on_http_errors: bool = True,
    request_timeout: int = 30,
    user_agent: str = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
):
    """
    Bulk scrape all country/year pages.

    Rules:
    - 404 => skip this country/year
    - no data on page => skip storing row
    - 403 or non-404 HTTP/network errors => stop whole run
    - sleep randomly between requests to reduce ban risk
    - take a long break every N requests
    """
    if sleep_range[0] < 0 or sleep_range[1] < sleep_range[0]:
        raise ValueError("sleep_range must be (min_seconds, max_seconds) with 0 <= min <= max")
    if long_break_every <= 0:
        raise ValueError("long_break_every must be > 0")
    if long_break_seconds < 0:
        raise ValueError("long_break_seconds must be >= 0")

    records = []
    global_match_ids = set()  # Fast global deduplicated access for next steps.
    match_ids_by_country_year = {}  # (country_id, year) -> set(match_ids)
    stats = {
        "total_requests": 0,
        "stored_rows": 0,
        "skipped_404": 0,
        "skipped_empty": 0,
        "long_breaks_taken": 0,
        "stopped": False,
        "stop_reason": None,
    }

    session = requests.Session()
    session.headers.update({"User-Agent": user_agent})

    try:
        for country_id, country_slug in countries.items():
            for year in range(start_year, end_year + 1):
                page_url = f"{BASE_URL}/country/{country_id}/{year}/{country_slug}.html"
                stats["total_requests"] += 1

                try:
                    resp = session.get(page_url, timeout=request_timeout)
                except requests.RequestException as exc:
                    stats["stopped"] = True
                    stats["stop_reason"] = f"Network/requests error at {page_url}: {exc}"
                    raise ScrapeStop(stats["stop_reason"])

                if resp.status_code == 404:
                    stats["skipped_404"] += 1
                elif resp.status_code == 403:
                    stats["stopped"] = True
                    stats["stop_reason"] = f"HTTP 403 at {page_url}. Server denied access."
                    raise ScrapeStop(stats["stop_reason"])
                elif resp.status_code != 200:
                    if stop_on_http_errors:
                        stats["stopped"] = True
                        stats["stop_reason"] = f"HTTP {resp.status_code} at {page_url}"
                        raise ScrapeStop(stats["stop_reason"])
                else:
                    row, has_data, page_match_ids = extract_country_year_info_from_html(
                        html=resp.text,
                        page_url=page_url,
                        country_id=country_id,
                        country_slug=country_slug,
                        year=year,
                    )

                    if has_data:
                        records.append(row)
                        stats["stored_rows"] += 1

                        key = (country_id, year)
                        match_set = set(page_match_ids)
                        match_ids_by_country_year[key] = match_set
                        global_match_ids.update(match_set)
                    else:
                        stats["skipped_empty"] += 1

                sleep_seconds = random.uniform(*sleep_range)
                time.sleep(sleep_seconds)

                if stats["total_requests"] % long_break_every == 0:
                    stats["long_breaks_taken"] += 1
                    print(
                        f"[PAUSE] Completed {stats['total_requests']} requests. "
                        f"Sleeping {long_break_seconds}s for cooldown..."
                    )
                    time.sleep(long_break_seconds)

    except ScrapeStop as stop_exc:
        print(f"[STOP] {stop_exc.message}")

    df = pd.DataFrame(records)
    if not df.empty:
        df = df.sort_values(["country_id", "year"]).reset_index(drop=True)

    result = {
        "df_country_year": df,
        "global_match_ids": global_match_ids,
        "match_ids_by_country_year": match_ids_by_country_year,
        "stats": stats,
    }
    return result


RUN_FULL_SCRAPE = True

if RUN_FULL_SCRAPE:
    bulk_result = scrape_all_afc_country_years(
        countries=afc_countries,
        start_year=1995,
        end_year=2026,
        sleep_range=(2, 5),
        long_break_every=50,
        long_break_seconds=60,
        stop_on_http_errors=True,
        request_timeout=30,
    )

    df_country_year = bulk_result["df_country_year"]
    global_match_ids = bulk_result["global_match_ids"]
    match_ids_by_country_year = bulk_result["match_ids_by_country_year"]
    stats = bulk_result["stats"]

    print("Stats:", stats)
    print("Rows in df_country_year:", len(df_country_year))
    print("Unique global match IDs:", len(global_match_ids))
    display(df_country_year.head())
else:
    print("Set RUN_FULL_SCRAPE = True to start bulk scraping.")

/tmp/ipykernel_1805/2044787217.py:66: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "scraped_at_utc": datetime.utcnow().isoformat(timespec="seconds"),


KeyboardInterrupt: 

In [ ]:
def build_country_year_tasks(countries: Dict[int, str], start_year: int, end_year: int):
    tasks = []
    for country_id, country_slug in countries.items():
        for year in range(start_year, end_year + 1):
            tasks.append((country_id, year, country_slug))
    return tasks


def parse_failed_country_year_from_stats(previous_stats: dict):
    """Extract failed URL + (country_id, year) from a stop reason like: HTTP 503 at <url>."""
    if not previous_stats:
        return None
    reason = previous_stats.get("stop_reason") or ""
    url_match = re.search(r"https?://\S+", reason)
    if not url_match:
        return None
    failed_url = url_match.group(0).rstrip(".")
    page_match = re.search(r"/country/(\d+)/(\d+)/([^/]+)\.html", failed_url)
    if not page_match:
        return None
    return {
        "failed_url": failed_url,
        "country_id": int(page_match.group(1)),
        "year": int(page_match.group(2)),
        "country_slug": page_match.group(3),
    }


def resume_afc_scrape(
    countries: Dict[int, str],
    existing_df: pd.DataFrame,
    existing_global_match_ids: set,
    existing_match_ids_by_country_year: dict,
    previous_stats: dict,
    start_year: int = 1995,
    end_year: int = 2026,
    sleep_range: Tuple[float, float] = (2, 5),
    long_break_every: int = 50,
    long_break_seconds: int = 60,
    stop_on_http_errors: bool = True,
    request_timeout: int = 30,
    resume_from_last_failed_url: bool = True,
    user_agent: str = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
):
    """Resume scraping using existing in-memory results, retrying the last failed page first."""
    if sleep_range[0] < 0 or sleep_range[1] < sleep_range[0]:
        raise ValueError("sleep_range must be (min_seconds, max_seconds) with 0 <= min <= max")
    if long_break_every <= 0:
        raise ValueError("long_break_every must be > 0")
    if long_break_seconds < 0:
        raise ValueError("long_break_seconds must be >= 0")

    tasks = build_country_year_tasks(countries, start_year, end_year)
    task_index = {(cid, year): idx for idx, (cid, year, _slug) in enumerate(tasks)}

    failed_info = parse_failed_country_year_from_stats(previous_stats)
    if resume_from_last_failed_url and failed_info:
        start_idx = task_index.get((failed_info["country_id"], failed_info["year"]), 0)
    else:
        start_idx = 0

    records = existing_df.to_dict("records") if existing_df is not None and not existing_df.empty else []
    global_match_ids = set(existing_global_match_ids or set())
    match_ids_by_country_year = {
        key: set(value)
        for key, value in (existing_match_ids_by_country_year or {}).items()
    }

    processed_keys = set(match_ids_by_country_year.keys())
    if existing_df is not None and not existing_df.empty:
        processed_keys.update(
            (int(row.country_id), int(row.year))
            for row in existing_df[["country_id", "year"]].itertuples(index=False)
        )

    resume_stats = {
        "total_requests": 0,
        "stored_rows": 0,
        "skipped_404": 0,
        "skipped_empty": 0,
        "long_breaks_taken": 0,
        "stopped": False,
        "stop_reason": None,
    }

    session = requests.Session()
    session.headers.update({"User-Agent": user_agent})

    try:
        for idx in range(start_idx, len(tasks)):
            country_id, year, country_slug = tasks[idx]
            key = (country_id, year)

            # Already-scraped rows are skipped to avoid duplicates on resume.
            if key in processed_keys:
                continue

            page_url = f"{BASE_URL}/country/{country_id}/{year}/{country_slug}.html"
            resume_stats["total_requests"] += 1

            try:
                resp = session.get(page_url, timeout=request_timeout)
            except requests.RequestException as exc:
                resume_stats["stopped"] = True
                resume_stats["stop_reason"] = f"Network/requests error at {page_url}: {exc}"
                raise ScrapeStop(resume_stats["stop_reason"])

            if resp.status_code == 404:
                resume_stats["skipped_404"] += 1
            elif resp.status_code == 403:
                resume_stats["stopped"] = True
                resume_stats["stop_reason"] = f"HTTP 403 at {page_url}. Server denied access."
                raise ScrapeStop(resume_stats["stop_reason"])
            elif resp.status_code != 200:
                if stop_on_http_errors:
                    resume_stats["stopped"] = True
                    resume_stats["stop_reason"] = f"HTTP {resp.status_code} at {page_url}"
                    raise ScrapeStop(resume_stats["stop_reason"])
            else:
                row, has_data, page_match_ids = extract_country_year_info_from_html(
                    html=resp.text,
                    page_url=page_url,
                    country_id=country_id,
                    country_slug=country_slug,
                    year=year,
                )

                if has_data:
                    records.append(row)
                    processed_keys.add(key)
                    resume_stats["stored_rows"] += 1

                    page_set = set(page_match_ids)
                    match_ids_by_country_year[key] = page_set
                    global_match_ids.update(page_set)
                else:
                    resume_stats["skipped_empty"] += 1

            time.sleep(random.uniform(*sleep_range))

            if resume_stats["total_requests"] % long_break_every == 0:
                resume_stats["long_breaks_taken"] += 1
                print(
                    f"[PAUSE] Resume completed {resume_stats['total_requests']} requests. "
                    f"Sleeping {long_break_seconds}s for cooldown..."
                )
                time.sleep(long_break_seconds)

    except ScrapeStop as stop_exc:
        print(f"[STOP] {stop_exc.message}")

    updated_df = pd.DataFrame(records)
    if not updated_df.empty:
        updated_df = updated_df.drop_duplicates(subset=["country_id", "year"], keep="first")
        updated_df = updated_df.sort_values(["country_id", "year"]).reset_index(drop=True)

    merged_stats = dict(previous_stats or {})
    count_keys = ["total_requests", "stored_rows", "skipped_404", "skipped_empty", "long_breaks_taken"]
    for key in count_keys:
        merged_stats[key] = merged_stats.get(key, 0) + resume_stats[key]
    merged_stats["stopped"] = resume_stats["stopped"]
    merged_stats["stop_reason"] = resume_stats["stop_reason"]
    merged_stats["resume_requests"] = resume_stats["total_requests"]
    merged_stats["resume_started_from_failed_url"] = bool(failed_info)

    return {
        "df_country_year": updated_df,
        "global_match_ids": global_match_ids,
        "match_ids_by_country_year": match_ids_by_country_year,
        "stats": merged_stats,
        "resume_stats": resume_stats,
        "resume_failed_info": failed_info,
    }


RUN_RESUME_SCRAPE = True

if RUN_RESUME_SCRAPE:
    resumed_result = resume_afc_scrape(
        countries=afc_countries,
        existing_df=df_country_year,
        existing_global_match_ids=global_match_ids,
        existing_match_ids_by_country_year=match_ids_by_country_year,
        previous_stats=stats,
        start_year=1995,
        end_year=2026,
        sleep_range=(2, 5),
        long_break_every=50,
        long_break_seconds=60,
        stop_on_http_errors=True,
        request_timeout=30,
        resume_from_last_failed_url=True,
    )

    bulk_result = resumed_result
    df_country_year = resumed_result["df_country_year"]
    global_match_ids = resumed_result["global_match_ids"]
    match_ids_by_country_year = resumed_result["match_ids_by_country_year"]
    stats = resumed_result["stats"]

    print("Resume stats:", resumed_result["resume_stats"])
    print("Merged stats:", stats)
    print("Rows in df_country_year:", len(df_country_year))
    print("Unique global match IDs:", len(global_match_ids))
    display(df_country_year.tail())
else:
    print("Set RUN_RESUME_SCRAPE = True in Cell 4 to continue from the last failed URL.")

/tmp/ipykernel_1805/2044787217.py:66: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "scraped_at_utc": datetime.utcnow().isoformat(timespec="seconds"),


[PAUSE] Resume completed 50 requests. Sleeping 60s for cooldown...
[PAUSE] Resume completed 100 requests. Sleeping 60s for cooldown...
[PAUSE] Resume completed 150 requests. Sleeping 60s for cooldown...
[PAUSE] Resume completed 200 requests. Sleeping 60s for cooldown...
[PAUSE] Resume completed 250 requests. Sleeping 60s for cooldown...
[PAUSE] Resume completed 300 requests. Sleeping 60s for cooldown...
[PAUSE] Resume completed 350 requests. Sleeping 60s for cooldown...
[PAUSE] Resume completed 400 requests. Sleeping 60s for cooldown...
[PAUSE] Resume completed 450 requests. Sleeping 60s for cooldown...
Resume stats: {'total_requests': 485, 'stored_rows': 485, 'skipped_404': 0, 'skipped_empty': 0, 'long_breaks_taken': 9, 'stopped': False, 'stop_reason': None}
Merged stats: {'total_requests': 1505, 'stored_rows': 1504, 'skipped_404': 0, 'skipped_empty': 0, 'long_breaks_taken': 29, 'stopped': False, 'stop_reason': None, 'resume_requests': 485, 'resume_started_from_failed_url': True}
Rows

,country_id,country_slug,year,page_url,average_height,average_age,match_count,match_ids,scraped_at_utc
1499,230,Myanmar,2022,https://national-football-teams.com/country/23...,1.75m,24.3,10,"[34640, 34788, 34848, 34911, 35280, 35337, 358...",2026-03-29T10:04:30
1500,230,Myanmar,2023,https://national-football-teams.com/country/23...,1.75m,24.3,11,"[35911, 36464, 36524, 36917, 36989, 37445, 385...",2026-03-29T10:04:33
1501,230,Myanmar,2024,https://national-football-teams.com/country/23...,1.75m,25.2,12,"[38615, 38754, 39029, 39154, 39549, 39640, 402...",2026-03-29T10:04:38
1502,230,Myanmar,2025,https://national-football-teams.com/country/23...,1.76m,25.2,5,"[42281, 41138, 41696, 42369, 42485]",2026-03-29T10:04:42
1503,230,Myanmar,2026,https://national-football-teams.com/country/23...,1.76m,25.8,1,[44212],2026-03-29T10:04:46


In [ ]:
from pathlib import Path
import json

# Save all current scrape outputs so the scraping step does not need to be rerun.
output_dir = Path("downloaded_files/scrape_exports")
output_dir.mkdir(parents=True, exist_ok=True)

# 1) Main country-year table
df_csv_path = output_dir / "afc_country_year_data.csv"
df_country_year.to_csv(df_csv_path, index=False, encoding="utf-8")

# 2) Global unique match IDs (one ID per row for easy reuse)
global_ids_csv_path = output_dir / "afc_global_match_ids.csv"
global_ids_df = pd.DataFrame({"match_id": sorted(global_match_ids)})
global_ids_df.to_csv(global_ids_csv_path, index=False, encoding="utf-8")

# 3) Country-year -> match IDs mapping in CSV form (normalized rows)
mapping_rows = []
for (country_id, year), id_set in sorted(match_ids_by_country_year.items()):
    for match_id in sorted(id_set):
        mapping_rows.append({
            "country_id": country_id,
            "year": year,
            "match_id": match_id,
        })

mapping_csv_path = output_dir / "afc_match_ids_by_country_year.csv"
pd.DataFrame(mapping_rows).to_csv(mapping_csv_path, index=False, encoding="utf-8")

# Optional JSON copy for direct dictionary reload later
mapping_json_path = output_dir / "afc_match_ids_by_country_year.json"
with mapping_json_path.open("w", encoding="utf-8") as f:
    json.dump(
        {f"{country_id}_{year}": sorted(list(id_set)) for (country_id, year), id_set in match_ids_by_country_year.items()},
        f,
        ensure_ascii=False,
        indent=2,
    )

print("Saved files:")
print("-", df_csv_path)
print("-", global_ids_csv_path)
print("-", mapping_csv_path)
print("-", mapping_json_path)
print("Rows in main CSV:", len(df_country_year))
print("Unique global match IDs:", len(global_match_ids))

Saved files:
- downloaded_files/scrape_exports/afc_country_year_data.csv
- downloaded_files/scrape_exports/afc_global_match_ids.csv
- downloaded_files/scrape_exports/afc_match_ids_by_country_year.csv
- downloaded_files/scrape_exports/afc_match_ids_by_country_year.json
Rows in main CSV: 1504
Unique global match IDs: 7555
